In [0]:
# Installs OpenAI SDK, Pydantic, NetworkX (for Tier 3 graph traversal), and Matplotlib
%pip install -q openai pydantic networkx matplotlib
%restart_python

In [0]:
# Creates a secure text box at the top of your notebook UI so your API key is NEVER saved to GitHub
dbutils.widgets.text("llm_api_key", "", "Paste API Key Here")
print("Look at the top of your notebook and paste your OpenAI or Gemini API key into 'Paste API Key Here'.")


In [0]:
import sys
sys.path.append("../src")

from agents import ThreeTierBenchmarkAgents

API_KEY = dbutils.widgets.get("llm_api_key")
assert API_KEY != "", "Please paste your API key into the 'Paste API Key Here' widget at the top!"

# Configure for OpenAI (default) OR uncomment the two Gemini lines below if using a Google AI Studio key:

# MODEL_NAME = "gemini-2.5-flash"
# BASE_URL = "https://googleapis.com"

MODEL_NAME = "gemini-4-hydrogen"
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

agents = ThreeTierBenchmarkAgents(spark=spark, api_key=API_KEY, model=MODEL_NAME, base_url=BASE_URL)

# Test 1 multi-hop causal question across all 3 tiers
test_q = "Which competitor brand do shoppers most frequently switch to when abandoning samsung?"

for name, runner in [
    ("Tier 1 (Raw Logs)", agents.run_tier1_raw_logs),
    ("Tier 2 (Semantic Layer)", agents.run_tier2_semantic_layer),
    ("Tier 3 (Ontology)", agents.run_tier3_ontology),
]:
    res = runner("TEST_01", test_q)
    print(f"=== {name} ===")
    print(f"Entity: {res.entity_answer} | Numeric: {res.numeric_answer} | Tokens: {res.prompt_tokens + res.completion_tokens} | Latency: {res.latency_sec}s")
    print(f"Code/Tool: {res.generated_code_or_tool}")
    if res.execution_error:
        print(f"Error: {res.execution_error}")
    print()


In [0]:
import time
import pandas as pd
from pyspark.sql import functions as F

# 1. Compute exact deterministic Ground Truth values from clean Silver & Ontology tables
def get_scalar(sql_str):
    return spark.sql(sql_str).collect()[0]["val"]

def get_top_edge(src, rel):
    row = spark.sql(f"""
        SELECT target_id, SUM(weight) AS w
        FROM workspace.default.ont_edges
        WHERE source_id = '{src}' AND relation_type = '{rel}'
        GROUP BY target_id ORDER BY w DESC LIMIT 1
    """).collect()[0]
    return row["target_id"].split(":")[1], float(row["w"])


gt_elec_rev = get_scalar("SELECT ROUND(SUM(session_revenue), 2) AS val FROM workspace.default.silver_sessions WHERE is_bot_suspect = false AND primary_category = 'electronics'")
gt_app_rev = get_scalar("SELECT ROUND(SUM(session_revenue), 2) AS val FROM workspace.default.silver_sessions WHERE is_bot_suspect = false AND primary_category = 'appliances'")
gt_comp_rev = get_scalar("SELECT ROUND(SUM(session_revenue), 2) AS val FROM workspace.default.silver_sessions WHERE is_bot_suspect = false AND primary_category = 'computers'")
gt_elec_direct = get_scalar("SELECT ROUND(SUM(CASE WHEN funnel_outcome = 'direct_purchase_no_cart' THEN 1 ELSE 0 END) * 100.0 / SUM(CASE WHEN purchase_count > 0 THEN 1 ELSE 0 END), 2) AS val FROM workspace.default.silver_sessions WHERE is_bot_suspect = false AND primary_category = 'electronics'")
gt_app_direct = get_scalar("SELECT ROUND(SUM(CASE WHEN funnel_outcome = 'direct_purchase_no_cart' THEN 1 ELSE 0 END) * 100.0 / SUM(CASE WHEN purchase_count > 0 THEN 1 ELSE 0 END), 2) AS val FROM workspace.default.silver_sessions WHERE is_bot_suspect = false AND primary_category = 'appliances'")

sam_comp, sam_sw = get_top_edge("brand:samsung", "SWITCHED_BRAND_TO")
app_comp, app_sw = get_top_edge("brand:apple", "SWITCHED_BRAND_TO")
xiao_comp, xiao_sw = get_top_edge("brand:xiaomi", "SWITCHED_BRAND_TO")
elec_ab_brand, elec_ab_cnt = get_top_edge("category:electronics", "ABANDONED_BRAND_CART")
app_ab_brand, app_ab_cnt = get_top_edge("category:appliances", "ABANDONED_BRAND_CART")

# 2. Balanced 10-Query Evaluation Suite (5 Descriptive BI + 5 Multi-Hop Causal)
benchmark_suite = [
    # Descriptive BI Queries
    {"query_id": "BI_01", "workload_type": "Descriptive_BI", "question": "What is the total purchase revenue in the electronics category excluding bot traffic?", "expected_num": float(gt_elec_rev), "expected_entity": "electronics"},
    {"query_id": "BI_02", "workload_type": "Descriptive_BI", "question": "What is the total purchase revenue in the appliances category excluding bot traffic?", "expected_num": float(gt_app_rev), "expected_entity": "appliances"},
    {"query_id": "BI_03", "workload_type": "Descriptive_BI", "question": "What is the total purchase revenue in the computers category excluding bot traffic?", "expected_num": float(gt_comp_rev), "expected_entity": "computers"},
    {"query_id": "BI_04", "workload_type": "Descriptive_BI", "question": "What percentage of purchasing sessions in electronics bypassed the cart via direct purchase?", "expected_num": float(gt_elec_direct), "expected_entity": "electronics"},
    {"query_id": "BI_05", "workload_type": "Descriptive_BI", "question": "What percentage of purchasing sessions in appliances bypassed the cart via direct purchase?", "expected_num": float(gt_app_direct), "expected_entity": "appliances"},
    # Multi-Hop Causal & Relational Queries
    {"query_id": "CAUSAL_01", "workload_type": "MultiHop_Causal", "question": "Which competitor brand do shoppers most frequently switch to when abandoning samsung?", "expected_num": sam_sw, "expected_entity": sam_comp},
    {"query_id": "CAUSAL_02", "workload_type": "MultiHop_Causal", "question": "Which competitor brand do shoppers most frequently switch to when abandoning apple?", "expected_num": app_sw, "expected_entity": app_comp},
    {"query_id": "CAUSAL_03", "workload_type": "MultiHop_Causal", "question": "Which competitor brand do shoppers most frequently switch to when abandoning xiaomi?", "expected_num": xiao_sw, "expected_entity": xiao_comp},
    {"query_id": "CAUSAL_04", "workload_type": "MultiHop_Causal", "question": "In the electronics category, which brand suffers the highest number of abandoned carts?", "expected_num": elec_ab_cnt, "expected_entity": elec_ab_brand},
    {"query_id": "CAUSAL_05", "workload_type": "MultiHop_Causal", "question": "In the appliances category, which brand suffers the highest number of abandoned carts?", "expected_num": app_ab_cnt, "expected_entity": app_ab_brand},
]

# 3. Execute all 3 tiers across all 10 benchmark queries (with a 4s pause to respect free-tier RPM limits)
records = []
for item in benchmark_suite:
    qid, wtype, q = item["query_id"], item["workload_type"], item["question"]
    print(f"Running {qid} ({wtype})...")
    for runner in [agents.run_tier1_raw_logs, agents.run_tier2_semantic_layer, agents.run_tier3_ontology]:
        res = runner(qid, q)
        num_ok = (
            res.numeric_answer is not None
            and item["expected_num"] > 0
            and abs(res.numeric_answer - item["expected_num"]) / item["expected_num"] <= 0.02
        )
        # For CAUSAL queries, require matching the target competitor/brand entity; for BI queries, require matching the numeric metric within 2%
        if wtype == "Descriptive_BI":
            is_accurate = int((res.execution_error is None) and num_ok)
        else:
            ent_ok = (res.entity_answer is not None) and (item["expected_entity"].lower() in res.entity_answer.lower())
            is_accurate = int((res.execution_error is None) and ent_ok)

        records.append({
            "query_id": qid,
            "workload_type": wtype,
            "tier": res.tier,
            "is_accurate": is_accurate,
            "numeric_answer": res.numeric_answer,
            "expected_num": item["expected_num"],
            "entity_answer": res.entity_answer,
            "expected_entity": item["expected_entity"],
            "total_tokens": res.prompt_tokens + res.completion_tokens,
            "latency_sec": res.latency_sec,
            "execution_error": res.execution_error,
            "generated_code_or_tool": res.generated_code_or_tool,
        })
        time.sleep(4)  # Stay safely within Gemini free-tier rate limits

df_results = pd.DataFrame(records)
spark.createDataFrame(df_results).write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_benchmark_results")

summary_df = (
    df_results.groupby(["tier", "workload_type"], as_index=False)
    .agg(
        accuracy_pct=("is_accurate", lambda s: round(s.mean() * 100, 1)),
        avg_tokens=("total_tokens", lambda s: round(s.mean(), 0)),
        avg_latency_sec=("latency_sec", lambda s: round(s.mean(), 2)),
    )
)
print("\n=== FINAL 3-TIER BENCHMARK SUMMARY ===")
print(summary_df.to_string(index=False))
